In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

## **Dynamic Multi-Asset Allocation using FactSet Data**

This project compares dynamic asset allocation strategies on a diversified 20-ETF universe.

At each rebalance date, I use only the previous 60 months of data to estimate the model and choose the portfolio weights. Then I apply those weights to the next month’s return. This creates a realistic out-of-sample test because the strategy never uses future information when making allocation decisions.

The main strategies are:

1. Equal-weight benchmark
2. 60/40 benchmark
3. Classical Markowitz optimization
4. Michaud resampling

The optional extension is Black-Litterman.

The data source is FactSet. I use monthly Total Return Gross levels and calculate monthly returns myself using percentage changes.

## Local configuration
Run Jupyter from the repository root. Place your licensed workbook under `data/`.

In [ ]:
from pathlib import Path
import os
from IPython.display import display
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
DATA_PATH = Path(os.environ.get("FMA_DATA_PATH", str(PROJECT_ROOT / "data/FactSet_20ETF_MultiAsset_TotalReturn_2008_2026.xlsx")))
SHEET_NAME = "Price History"
if not DATA_PATH.exists():
    raise FileNotFoundError("Supply your licensed workbook; see data/README.md")


## **Define the ETF universe**

Use 20 ETFs to represent a diversified multi-asset universe.
The assets cover equities, bonds, credit, gold, commodities, real estate, and cash-like ETFs,
so the project studies asset allocation across markets, not only stocks.

In [ ]:
asset_classes = {
    "SPY": "US Equity - Large Cap",
    "QQQ": "US Equity - Growth / Nasdaq",
    "IWM": "US Equity - Small Cap",
    "VGK": "Europe Equity",
    "EFA": "Developed ex-US Equity",
    "EEM": "Emerging Markets Equity",
    "EWJ": "Japan Equity",
    "SHY": "Short-Term US Treasury",
    "IEF": "Intermediate-Term US Treasury",
    "TLT": "Long-Term US Treasury",
    "AGG": "US Aggregate Bonds",
    "LQD": "Investment Grade Credit",
    "HYG": "High Yield Credit",
    "EMB": "Emerging Market Bonds",
    "TIP": "Inflation-Linked Bonds",
    "GLD": "Gold",
    "DBC": "Broad Commodities",
    "VNQ": "Real Estate / REIT",
    "BIL": "Cash Proxy - T-Bill",
    "SHV": "Cash-like Short Treasury"
}

tickers = list(asset_classes.keys())

universe_table = pd.DataFrame({
    "Ticker": tickers,
    "Asset Class": [asset_classes[t] for t in tickers]
})

universe_table

## **Loading the dataset**


In [ ]:
levels = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, header=15)

levels.head(20)

## **Clean column names**

In [ ]:
# Extract ETF tickers from the long FactSet column names

def extract_ticker(column):
    match = re.search(r"\(([A-Z]+)-US\)", str(column))
    if match:
        return match.group(1)
    return column

levels = levels.rename(columns=extract_ticker)

# Remove the Composite column because it is not an investable ETF

levels = levels.drop(columns=["Composite"], errors="ignore")

# Convert the Date column and use it as the index

levels["Date"] = pd.to_datetime(levels["Date"])
levels = levels.set_index("Date").sort_index()

# Keep only the selected ETFs and the clean common sample period

levels = levels.loc["2008-01-01":"2026-05-31", tickers]

levels.head()

## **Monthly returns**

In [ ]:
returns = (levels / levels.shift(1) - 1).dropna()
returns.head()

In [ ]:
# Check cleaned dataset

print("Start date:", levels.index.min())
print("End date:", levels.index.max())
print("Number of monthly observations:", len(levels))
print("Number of ETFs:", levels.shape[1])
print("Total missing values:", levels.isna().sum().sum())

## **Annualization**

In [ ]:
# Annualization from monthly data (12 months per year)

MONTHS_PER_YEAR = 12

mu = returns.mean() * MONTHS_PER_YEAR           # expected return vector (annual)
Sigma = returns.cov() * MONTHS_PER_YEAR         # covariance matrix (annual)
sigma = np.sqrt(np.diag(Sigma))              # annualized volatilities

In [ ]:
stats = pd.DataFrame({
    "Asset class": [asset_classes[t] for t in returns.columns],
    "Exp. Return (%)": (mu * 100).round(2).values,
    "Ann. Vol (%)": (sigma * 100).round(2),
    "Sharpe ": (mu.values / sigma).round(2),
}, index=returns.columns)

stats

In [ ]:
# Normalize Total Return levels to 100 for visual comparison

normalized_levels = levels / levels.iloc[0] * 100

normalized_levels.plot(figsize=(12, 6), title="ETF Performance Normalized to 100")

plt.ylabel("Index (base 100)")
plt.xlabel("Date")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## **Correlation matrix**

The correlation matrix shows how ETF returns move together.
It is important because diversification is stronger when assets are not perfectly correlated.
This also matters for Markowitz because the optimizer uses the covariance matrix, which combines volatility and correlation.

In [ ]:
# Correlation matrix of monthly returns
# This helps us understand diversification across ETFs

corr = returns.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)

plt.title("Correlation Matrix of Monthly ETF Returns")
plt.tight_layout()
plt.show()

## **Benchmark Portfolios**

###**1. equal-weight benchmark**

For each month, portfolio return = weighted average of ETF returns.
Since all weights are equal, every ETF contributes equally.

In [ ]:
n_assets = len(returns.columns)
equal_weights = np.repeat(1 / n_assets, n_assets)

equal_weight_returns = pd.Series(
    returns @ equal_weights,
    index=returns.index,
    name="Equal Weight"
)

equal_weight_returns.head()

###**2. 60/40 benchmark**

I use a simple 60/40 benchmark as a balanced portfolio.  
The 60% equity part includes ETFs such as SPY, QQQ, IWM, EFA, EEM, VGK, and EWJ.  
The 40% bond and cash-like part includes ETFs such as AGG, SHY, IEF, TLT, LQD, TIP, BIL, and SHV.

In [ ]:
# 60% is allocated to equity ETFs and 40% to bond/cash ETFs

equity_assets = ["SPY", "QQQ", "IWM", "VGK", "EFA", "EEM", "EWJ"]
bond_cash_assets = ["SHY", "IEF", "TLT", "AGG", "LQD", "BIL", "SHV"]

equity_basket_returns = returns[equity_assets].mean(axis=1)
bond_cash_basket_returns = returns[bond_cash_assets].mean(axis=1)

benchmark_60_40_returns = (
    0.60 * equity_basket_returns +
    0.40 * bond_cash_basket_returns
)

benchmark_60_40_returns.name = "60/40"

benchmark_60_40_returns.head()

In [ ]:
# Combine benchmark return series

benchmark_returns = pd.DataFrame({
    "Equal Weight": equal_weight_returns,
    "60/40": benchmark_60_40_returns
})

benchmark_returns.head()

In [ ]:
# Function to calculate portfolio performance statistics

def performance_stats(portfolio_returns):
    ann_return = portfolio_returns.mean() * 12
    ann_vol = portfolio_returns.std() * np.sqrt(12)
    sharpe = ann_return / ann_vol

    cumulative_growth = (1 + portfolio_returns).cumprod()
    running_max = cumulative_growth.cummax()
    drawdown = cumulative_growth / running_max - 1
    max_drawdown = drawdown.min()

    return pd.Series({
        "Ann. Return (%)": ann_return * 100,
        "Ann. Vol (%)": ann_vol * 100,
        "Sharpe ": sharpe,
        "Max Drawdown (%)": max_drawdown * 100
    })


benchmark_stats = benchmark_returns.apply(performance_stats).T.round(2)

benchmark_stats

In [ ]:
# Plot cumulative performance of benchmark portfolios

benchmark_growth = (1 + benchmark_returns).cumprod()

benchmark_growth.plot(figsize=(10, 5), title="Benchmark Portfolio Performance")

plt.ylabel("Growth of 1 Dollar")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

# **Markowitz Minimum Variance**



The Markowitz minimum-variance strategy tries to find the portfolio with the lowest possible risk. Instead of looking at each ETF separately, it uses the covariance matrix to understand how the ETFs move together.

 The portfolio is long-only, fully invested, and each ETF is capped at 20% to avoid extreme concentration.

After the weights are computed, they are applied to the next month’s returns. This makes the strategy dynamic and out-of-sample, because only past information is used when forming each portfolio.

### **Rolling Out-of-Sample Markowitz Optimization**


In this section, I run the Markowitz strategy in a rolling out-of-sample framework. At each monthly rebalance date, I use only the previous **60 months** of returns to estimate the covariance matrix. Then I compute the minimum-variance portfolio weights under the constraints: long-only, fully invested, and maximum **20%** weight per ETF.

After the weights are calculated, they are applied to the **next month’s realized return**. Then the window moves forward by one month and the process is repeated.

This structure avoids look-ahead bias because the model never uses future returns when choosing the portfolio weights.


In [ ]:
# Optimization settings

WINDOW = 60          # use previous 60 months to estimate inputs
MAX_WEIGHT = 0.20    # maximum 20% in each ETF
MONTHS_PER_YEAR = 12

In [ ]:
# Markowitz minimum-variance portfolio
# Objective: minimize portfolio variance w'Σw
# Constraints: weights sum to 1, long-only, max weight cap

def optimize_min_variance(cov_matrix, max_weight=0.20):
    n_assets = cov_matrix.shape[0]
    initial_weights = np.repeat(1 / n_assets, n_assets)

    bounds = [(0, max_weight) for _ in range(n_assets)]
    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}

    def portfolio_variance(w):
        return w @ cov_matrix @ w

    result = minimize(
        portfolio_variance,
        initial_weights,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if result.success:
        return result.x
    else:
        return initial_weights

In [ ]:
# Rolling out-of-sample Markowitz backtest
# At each month, use only the previous 60 months to estimate the covariance matrix
from scipy.optimize import minimize

markowitz_returns = []
markowitz_weights = []
backtest_dates = []

for i in range(WINDOW, len(returns)):
    estimation_window = returns.iloc[i-WINDOW:i]
    next_month_return = returns.iloc[i]

    cov_matrix = estimation_window.cov() * MONTHS_PER_YEAR

    weights = optimize_min_variance(cov_matrix.values, max_weight=MAX_WEIGHT)

    portfolio_return = weights @ next_month_return.values

    markowitz_returns.append(portfolio_return)
    markowitz_weights.append(weights)
    backtest_dates.append(returns.index[i])

markowitz_returns = pd.Series(
    markowitz_returns,
    index=backtest_dates,
    name="Markowitz Min Variance"
)

markowitz_weights = pd.DataFrame(
    markowitz_weights,
    index=backtest_dates,
    columns=returns.columns
)

markowitz_returns.head()

###### **Comparing Benchmarks and Markowitz**

In [ ]:
# Align benchmark returns with the Markowitz backtest period

comparison_returns = pd.DataFrame({
    "Equal Weight": equal_weight_returns,
    "60/40": benchmark_60_40_returns,
    "Markowitz Min Variance": markowitz_returns
}).dropna()

comparison_returns.head()

In [ ]:
# Plot cumulative performance of benchmarks and Markowitz

comparison_growth = (1 + comparison_returns).cumprod()

comparison_growth.plot(figsize=(11, 6), title="Out-of-Sample Portfolio Performance")

plt.ylabel("Growth of 1 Dollar")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Performance statistics for each strategy
comparison_stats = comparison_returns.apply(performance_stats).T.round(2)

comparison_stats

### **Check Markowitz weights**

In [ ]:
# Average Markowitz weights over the backtest period

average_markowitz_weights = markowitz_weights.mean().sort_values(ascending=False)

average_markowitz_weights

In [ ]:
# Plot average Markowitz weights over the out of sample basket

average_markowitz_weights.plot(kind="bar", figsize=(10, 5), title="Average Markowitz Portfolio Weights")

plt.ylabel("Average Weight")
plt.xlabel("ETF")
plt.tight_layout()
plt.show()

## **Robustness Check: 40% Weight Cap**

In [ ]:
# Robustness check: Markowitz with 40% maximum weight per ETF

MAX_WEIGHT_40 = 0.40

markowitz_40_returns = []
markowitz_40_weights = []
backtest_dates_40 = []

for i in range(WINDOW, len(returns)):
    estimation_window = returns.iloc[i-WINDOW:i]
    next_month_return = returns.iloc[i]

    cov_matrix = estimation_window.cov() * MONTHS_PER_YEAR

    weights = optimize_min_variance(cov_matrix.values, max_weight=MAX_WEIGHT_40)

    portfolio_return = weights @ next_month_return.values

    markowitz_40_returns.append(portfolio_return)
    markowitz_40_weights.append(weights)
    backtest_dates_40.append(returns.index[i])

markowitz_40_returns = pd.Series(
    markowitz_40_returns,
    index=backtest_dates_40,
    name="Markowitz Min Variance 40% Cap"
)

markowitz_40_weights = pd.DataFrame(
    markowitz_40_weights,
    index=backtest_dates_40,
    columns=returns.columns
)

markowitz_40_returns.head()

In [ ]:
cap_comparison_returns = pd.DataFrame({
    "Markowitz 20% Cap": markowitz_returns,
    "Markowitz 40% Cap": markowitz_40_returns
}).dropna()

cap_comparison_stats = cap_comparison_returns.apply(performance_stats).T.round(2)

cap_comparison_stats

In [ ]:
cap_comparison_growth = (1 + cap_comparison_returns).cumprod()

cap_comparison_growth.plot(figsize=(10, 5), title="Markowitz Weight Cap Sensitivity")

plt.ylabel("Growth of 1 Dollar")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

As a robustness check, I compare the baseline 20% maximum weight constraint with a looser 40% cap. The 40% cap gives the optimizer more freedom to concentrate in low-risk assets. As expected, this reduces volatility and maximum drawdown substantially, but it also lowers annual return. This confirms that the weight cap has an important effect on the risk-return profile of the Markowitz minimum-variance strategy.

## **Rolling Michaud Resampled Minimum-Variance Strategy**



The Michaud resampled strategy is used as a more robust version of the Markowitz minimum-variance portfolio. A standard Markowitz optimizer can be very sensitive to estimation error, because it relies on one estimated covariance matrix.

To reduce this sensitivity, I simulate many alternative return histories based on the mean and covariance estimated from the previous 60 months. For each simulated sample, I solve the same minimum-variance optimization problem. Then I average the optimized weights across all simulations.

This produces smoother and more stable portfolio weights. The strategy remains rolling and out-of-sample, because at each month only the previous 60 months are used to estimate the inputs.


In [ ]:
def michaud_resampled_min_variance(estimation_window, n_simulations=100, max_weight=0.20):
    window_length = estimation_window.shape[0]

    # Estimate mean and covariance from the past window
    mu_monthly = estimation_window.mean().values
    cov_monthly = estimation_window.cov().values

    simulated_weights = []

    for _ in range(n_simulations):

        # Simulate an alternative return history with the same mean and covariance
        simulated_returns = np.random.multivariate_normal(
            mean=mu_monthly,
            cov=cov_monthly,
            size=window_length
        )

        # Estimate covariance from the simulated data
        simulated_cov = np.cov(simulated_returns, rowvar=False) * MONTHS_PER_YEAR

        # Optimize the simulated sample using the same Markowitz minimum-variance rule
        weights = optimize_min_variance(
            simulated_cov,
            max_weight=max_weight
        )

        simulated_weights.append(weights)

    # Average the optimized weights across all simulations
    resampled_weights = np.mean(simulated_weights, axis=0)

    # Numerical correction: make sure weights sum to 1
    resampled_weights = resampled_weights / resampled_weights.sum()

    return resampled_weights

In [ ]:
# Rolling out-of-sample Michaud resampling backtest
# At each month, use the previous 60 months, calculate resampled weights,
# and apply those weights to the next month's real return.

np.random.seed(42)

N_SIMULATIONS = 100

michaud_returns = []
michaud_weights = []
michaud_dates = []

for i in range(WINDOW, len(returns)):
    estimation_window = returns.iloc[i-WINDOW:i]
    next_month_return = returns.iloc[i]

    weights = michaud_resampled_min_variance(
        estimation_window,
        n_simulations=N_SIMULATIONS,
        max_weight=MAX_WEIGHT
    )

    portfolio_return = weights @ next_month_return.values

    michaud_returns.append(portfolio_return)
    michaud_weights.append(weights)
    michaud_dates.append(returns.index[i])

michaud_returns = pd.Series(
    michaud_returns,
    index=michaud_dates,
    name="Michaud Resampled Min Variance"
)

michaud_weights = pd.DataFrame(
    michaud_weights,
    index=michaud_dates,
    columns=returns.columns
)

michaud_returns.head()

In [ ]:
# Compare benchmarks, Markowitz, and Michaud on the same out-of-sample period

final_comparison_returns = pd.DataFrame({
    "Equal Weight": equal_weight_returns,
    "60/40": benchmark_60_40_returns,
    "Markowitz Min Variance": markowitz_returns,
    "Michaud Resampled": michaud_returns
}).dropna()

final_comparison_returns.head()

In [ ]:
# Performance statistics for all strategies

final_comparison_stats = final_comparison_returns.apply(performance_stats).T.round(2)

final_comparison_stats

Michaud resampling does not dramatically outperform Markowitz in this result because both strategies use the same minimum-variance objective and the same constraints. Since the optimizer mainly selects defensive low-risk ETFs, the resampled portfolios remain close to the Markowitz solution. Michaud’s role is mainly to reduce sensitivity to estimation error and improve robustness, not necessarily to generate much higher returns. In this backtest, Michaud slightly improves annual return and Sharpe ratio, but the overall performance remains very similar to Markowitz.


In [ ]:
# Plot cumulative performance of all strategies

final_comparison_growth = (1 + final_comparison_returns).cumprod()

final_comparison_growth.plot(
    figsize=(11, 6),
    title="Out-of-Sample Strategy Comparison"
)

plt.ylabel("Growth of 1 Dollar")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Average Michaud weights over the backtest period

average_michaud_weights = michaud_weights.mean().sort_values(ascending=False)

average_michaud_weights

In [ ]:
# Plot average Michaud portfolio weights

average_michaud_weights.plot(
    kind="bar",
    figsize=(10, 5),
    title="Average Michaud Resampled Portfolio Weights"
)

plt.ylabel("Average Weight")
plt.xlabel("ETF")
plt.tight_layout()
plt.show()

In [ ]:
# Compute drawdowns for all strategies
#A drawdown shows how far the portfolio is below its previous peak.

def compute_drawdown(return_series):
    cumulative_growth = (1 + return_series).cumprod()
    running_max = cumulative_growth.cummax()
    drawdown = cumulative_growth / running_max - 1
    return drawdown


drawdowns = final_comparison_returns.apply(compute_drawdown)

drawdowns.plot(figsize=(11, 6), title="Drawdown Comparison")

plt.ylabel("Drawdown")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

### **Markowitz vs Michaud weight comparison**

In [ ]:
# Compare average weights of Markowitz and Michaud strategies

average_weights_comparison = pd.DataFrame({
    "Markowitz": markowitz_weights.mean(),
    "Michaud": michaud_weights.mean()
})

average_weights_comparison.sort_values("Markowitz", ascending=False)

In [ ]:
average_weights_comparison.plot(
    kind="bar",
    figsize=(12, 5),
    title="Average Portfolio Weights: Markowitz vs Michaud"
)

plt.ylabel("Average Weight")
plt.xlabel("ETF")
plt.tight_layout()
plt.show()

This chart compares the average portfolio weights of Markowitz and Michaud. Both methods allocate mainly to defensive bond and cash-like ETFs such as SHY, BIL, SHV, AGG, IEF, and TIP. The weights are very similar because both strategies have the same minimum-variance objective. Michaud does not change the main allocation pattern, but it makes the optimization more robust by averaging weights across simulated samples.


In [ ]:
# Portfolio turnover
# Turnover measures how much portfolio weights change from one month to the next

def compute_turnover(weights):
    return weights.diff().abs().sum(axis=1).dropna()

markowitz_turnover = compute_turnover(markowitz_weights)
michaud_turnover = compute_turnover(michaud_weights)

turnover_comparison = pd.DataFrame({
    "Markowitz": markowitz_turnover,
    "Michaud": michaud_turnover
}).dropna()

turnover_comparison.mean()

In [ ]:
turnover_comparison.plot(figsize=(11, 5), title="Monthly Portfolio Turnover")

plt.ylabel("Turnover")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

### **Rolling Markowitz Max Sharpe Strategy**

In this section, I implement a rolling Markowitz maximum Sharpe strategy. Unlike the minimum-variance strategy, this method uses both expected returns and risk. At each month, I estimate expected returns and the covariance matrix using the previous 60 months. Then I choose the weights that maximize the Sharpe ratio, subject to the same constraints: long-only, fully invested, and maximum 20% per ETF. Since the optimizer minimizes functions, I minimize the negative Sharpe ratio, which is equivalent to maximizing the Sharpe ratio.

In [ ]:
# Maximum Sharpe optimizer
# Objective: maximize portfolio Sharpe ratio
# Constraints: weights sum to 1, long-only, max weight cap

def optimize_max_sharpe(expected_returns, cov_matrix, max_weight=0.20, risk_free_rate=0.0):
    n_assets = len(expected_returns)
    initial_weights = np.repeat(1 / n_assets, n_assets)

    bounds = [(0, max_weight) for _ in range(n_assets)]
    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}

    def negative_sharpe(w):
        portfolio_return = w @ expected_returns
        portfolio_volatility = np.sqrt(w @ cov_matrix @ w)

        if portfolio_volatility == 0:
            return 1e6

        return -(portfolio_return - risk_free_rate) / portfolio_volatility

    result = minimize(
        negative_sharpe,
        initial_weights,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if result.success:
        return result.x
    else:
        return initial_weights

In [ ]:
# Rolling Markowitz maximum-Sharpe backtest
# Expected returns are estimated using historical sample means from the past 60 months.

markowitz_sharpe_returns = []
markowitz_sharpe_weights = []
markowitz_sharpe_dates = []

for i in range(WINDOW, len(returns)):
    estimation_window = returns.iloc[i-WINDOW:i]
    next_month_return = returns.iloc[i]

    # Estimate annual expected returns and covariance from the previous 60 months
    expected_returns = estimation_window.mean().values * MONTHS_PER_YEAR
    cov_matrix = estimation_window.cov().values * MONTHS_PER_YEAR

    # Optimize maximum Sharpe portfolio
    weights = optimize_max_sharpe(
        expected_returns=expected_returns,
        cov_matrix=cov_matrix,
        max_weight=MAX_WEIGHT,
        risk_free_rate=0.0
    )

    # Apply weights to next month's real return
    portfolio_return = weights @ next_month_return.values

    markowitz_sharpe_returns.append(portfolio_return)
    markowitz_sharpe_weights.append(weights)
    markowitz_sharpe_dates.append(returns.index[i])

markowitz_sharpe_returns = pd.Series(
    markowitz_sharpe_returns,
    index=markowitz_sharpe_dates,
    name="Markowitz Max Sharpe"
)

markowitz_sharpe_weights = pd.DataFrame(
    markowitz_sharpe_weights,
    index=markowitz_sharpe_dates,
    columns=returns.columns
)

markowitz_sharpe_returns.head()

In [ ]:
markowitz_sharpe_stats = performance_stats(markowitz_sharpe_returns).round(2)

markowitz_sharpe_stats

In [ ]:
comparison_with_max_sharpe_returns = pd.DataFrame({
    "Equal Weight": equal_weight_returns,
    "60/40": benchmark_60_40_returns,
    "Markowitz Min Variance": markowitz_returns,
    "Michaud Resampled": michaud_returns,
    "Markowitz Max Sharpe": markowitz_sharpe_returns
}).dropna()

comparison_with_max_sharpe_stats = comparison_with_max_sharpe_returns.apply(performance_stats).T.round(2)

comparison_with_max_sharpe_stats

In [ ]:
comparison_with_max_sharpe_growth = (1 + comparison_with_max_sharpe_returns).cumprod()

comparison_with_max_sharpe_growth.plot(
    figsize=(12, 6),
    title="Comparison Including Markowitz Max Sharpe"
)

plt.ylabel("Growth of 1 Dollar")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
average_markowitz_sharpe_weights = markowitz_sharpe_weights.mean().sort_values(ascending=False)

average_markowitz_sharpe_weights

In [ ]:
average_markowitz_sharpe_weights.plot(
    kind="bar",
    figsize=(10, 5),
    title="Average Markowitz Max Sharpe Portfolio Weights"
)

plt.ylabel("Average Weight")
plt.xlabel("ETF")
plt.tight_layout()
plt.show()

## **Optional Extension: Black-Litterman Strategy**





Black-Litterman is used as an optional extension to combine a neutral prior with investor or model-based views. In this project, the prior is based on a neutral equal-weight portfolio during the rolling backtest, and the views are generated dynamically from past 12-month momentum.

At each rebalancing date, I use only the previous 60 months of returns to estimate risk and create the Black-Litterman inputs. The model then combines the prior and the momentum views to produce posterior expected returns. These posterior returns are used in a maximum-Sharpe optimizer with the same practical constraints: long-only, fully invested, and a 20% maximum weight per ETF.

This keeps the strategy dynamic and out-of-sample while allowing expected returns to be adjusted by systematic views.


In [ ]:
# Black-Litterman-specific assumptions.
# The general project settings WINDOW, MAX_WEIGHT, and MONTHS_PER_YEAR are already defined earlier.

RISK_AVERSION = 2.5
TAU = 0.05
SHRINKAGE = 0.50

asset_list = pd.Index(returns.columns)
n_assets = len(asset_list)

# Basic checks before adding the Black-Litterman extension.
assert n_assets == 20, f"Expected 20 ETFs, but found {n_assets}."
assert returns.isna().sum().sum() == 0, "returns contains missing values."
assert returns.index.is_monotonic_increasing, "returns index is not sorted ascending."


assert "optimize_max_sharpe" in globals(), "Run the Markowitz Max Sharpe optimizer cell first."
assert "performance_stats" in globals(), "Run the performance_stats cell first."
assert "compute_drawdown" in globals(), "Run the compute_drawdown cell first."
assert "compute_turnover" in globals(), "Run the compute_turnover cell first."

## Black–Litterman View Generation

* Views are generated dynamically at each rebalance date.
* Only past data is used; no future information is included.
* Rolling estimation window: **60 months**.
* Recent performance window: **12 months**.
* Number of views generated at each rebalance date: **2 views**.

### Step 1: Rank ETFs by recent performance

* For each ETF, calculate the compounded return over the last **12 months**.
* Select:

  * **Strongest ETF** = ETF with the highest 12-month return.
  * **Weakest ETF** = ETF with the lowest 12-month return.
* These two ETFs are used to create the Black–Litterman views.

### View 1: Relative View

* Type: **Relative view**.
* Logic:

  * Strongest ETF is expected to outperform weakest ETF.
* P matrix structure:

  * Strongest ETF: **+1**
  * Weakest ETF: **-1**
  * All other ETFs: **0**
* View value:

  * Start from the return difference between strongest and weakest ETF.
  * Use only **50%** of this difference.
* Cap:

  * Minimum relative view: **1%**
  * Maximum relative view: **6%**
* Reason:

  * Full return differences can be too aggressive.
  * The **50% scaling** and **6% cap** prevent extreme views.

### Relative View Confidence

* Confidence depends on the size of the return difference:

  * Return difference < **3%** → confidence = **60%**
  * Return difference between **3% and 8%** → confidence = **70%**
  * Return difference ≥ **8%** → confidence = **80%**
* Logic:

  * Small difference → weaker signal → lower confidence.
  * Large difference → stronger signal → higher confidence.

### View 2: Absolute View

* Type: **Absolute view**.
* Asset used:

  * Only the **strongest ETF**.
* Logic:

  * Assign an expected annual return to the strongest ETF.
* View value:

  * **50% historical annualized return**
  * plus **50% prior-implied return**
* Shrinkage:

  * Shrinkage weight = **50%**
* Reason:

  * Historical mean returns are noisy.
  * Shrinking toward the prior makes the view more conservative.

### Absolute View Cap and Confidence

* Cap:

  * Minimum absolute view: **-5%**
  * Maximum absolute view: **15%**
* Reason:

  * Prevents unrealistically negative or high expected-return forecasts.
  * Avoids overly concentrated portfolio weights.
* Confidence:

  * Fixed confidence = **60%**
* Reason:

  * Predicting the exact return of one ETF is uncertain.

### Omega Matrix

* Omega represents view uncertainty.
* Omega is calculated from:

  * View variance
  * Confidence level
* Interpretation:

  * Higher confidence → smaller Omega → stronger effect on posterior returns.
  * Lower confidence → larger Omega → weaker effect on posterior returns.
* Numerical stability:

  * Minimum Omega floor = **1e-6**.




## **Dynamic Black-Litterman views**

In [ ]:
def create_dynamic_individual_bl_views(estimation_window, asset_list, Sigma, pi):
    """
    Create two dynamic Black-Litterman views using only past data.

    View 1:
    The ETF with the strongest trailing 12-month return outperforms
    the ETF with the weakest trailing 12-month return.

    View 2:
    The strongest momentum ETF receives an absolute expected-return view,
    but this view is shrunk toward the prior-implied return.
    """

    asset_list = pd.Index(asset_list)
    n_assets = len(asset_list)

    assert len(estimation_window) >= 12, "At least 12 months are needed for the momentum view."
    assert Sigma.shape == (n_assets, n_assets), "Sigma has the wrong shape."
    assert pi.shape == (n_assets,), "pi has the wrong shape."

    # Use only the last 12 months inside the rolling estimation window.
    momentum_12m = (1 + estimation_window.tail(12)).prod() - 1

    top_asset = momentum_12m.idxmax()
    bottom_asset = momentum_12m.idxmin()

    top_idx = asset_list.get_loc(top_asset)
    bottom_idx = asset_list.get_loc(bottom_asset)

    top_momentum = momentum_12m.loc[top_asset]
    bottom_momentum = momentum_12m.loc[bottom_asset]
    spread_1 = top_momentum - bottom_momentum

    # Relative view: top momentum ETF outperforms bottom momentum ETF.
    P1 = np.zeros(n_assets)
    P1[top_idx] = 1.0
    P1[bottom_idx] = -1.0

    # Use only half of the observed spread and cap it to avoid unrealistic views.
    Q1 = np.clip(0.5 * abs(spread_1), 0.01, 0.06)

    view_variance_1 = float(P1 @ Sigma @ P1.T)
    view_variance_1 = max(view_variance_1, 1e-8)

    # Larger spreads are treated as stronger signals.
    if abs(spread_1) < 0.03:
        confidence_1 = 0.60
    elif abs(spread_1) < 0.08:
        confidence_1 = 0.70
    else:
        confidence_1 = 0.80

    omega_1 = ((1 - confidence_1) / confidence_1) * view_variance_1
    omega_1 = max(float(omega_1), 1e-6)

    # Absolute view for the same strongest momentum ETF.
    P2 = np.zeros(n_assets)
    P2[top_idx] = 1.0

    mu_hist_top = estimation_window[top_asset].mean() * MONTHS_PER_YEAR
    mu_prior_top = pi[top_idx]

    # Shrink the historical estimate toward the prior because historical means are noisy.
    Q2 = SHRINKAGE * mu_hist_top + (1 - SHRINKAGE) * mu_prior_top
    Q2 = np.clip(Q2, -0.05, 0.15)

    view_variance_2 = float(P2 @ Sigma @ P2.T)
    view_variance_2 = max(view_variance_2, 1e-8)

    T_years = len(estimation_window) / MONTHS_PER_YEAR
    confidence_2 = 0.60

    omega_2 = view_variance_2 / T_years
    omega_2 = ((1 - confidence_2) / confidence_2) * omega_2
    omega_2 = max(float(omega_2), 1e-6)

    P = np.vstack([P1, P2])
    Q = np.array([Q1, Q2])
    Omega = np.diag([omega_1, omega_2])

    views_table = pd.DataFrame({
        "View name": [
            "Top momentum outperforms bottom momentum",
            "Absolute view on strongest momentum ETF"
        ],
        "View type": [
            "Relative",
            "Absolute"
        ],
        "Top asset": [
            top_asset,
            top_asset
        ],
        "Bottom asset": [
            bottom_asset,
            None
        ],
        "Top asset 12M momentum": [
            top_momentum,
            top_momentum
        ],
        "Bottom asset 12M momentum": [
            bottom_momentum,
            None
        ],
        "Momentum spread": [
            spread_1,
            None
        ],
        "Q annual view": [
            Q1,
            Q2
        ],
        "Historical annualized return": [
            None,
            mu_hist_top
        ],
        "Prior-implied return": [
            None,
            mu_prior_top
        ],
        "Shrinkage": [
            None,
            SHRINKAGE
        ],
        "Confidence": [
            confidence_1,
            confidence_2
        ],
        "View variance": [
            view_variance_1,
            view_variance_2
        ],
        "Omega value": [
            omega_1,
            omega_2
        ],
        "Explanation": [
            f"{top_asset} has the highest trailing 12-month return and {bottom_asset} has the lowest.",
            f"The expected return for {top_asset} is shrunk toward its prior-implied return."
        ]
    })


    assert P.shape == (2, n_assets), "P has the wrong shape."
    assert Q.shape == (2,), "Q has the wrong shape."
    assert Omega.shape == (2, 2), "Omega has the wrong shape."
    assert np.all(np.diag(Omega) > 0), "Omega diagonal values must be positive."

    assert np.isclose(P[0].sum(), 0.0), "Relative view row must sum to zero."
    assert np.sum(P[0] == 1.0) == 1, "Relative view must have one +1."
    assert np.sum(P[0] == -1.0) == 1, "Relative view must have one -1."

    assert np.isclose(P[1].sum(), 1.0), "Absolute view row must sum to one."
    assert np.sum(P[1] == 1.0) == 1, "Absolute view must have one +1."
    assert np.sum(P[1] != 0.0) == 1, "Absolute view must select only one ETF."

    return P, Q, Omega, views_table

## **Black-Litterman posterior**

In [ ]:
def black_litterman_posterior(pi, Sigma, P, Q, Omega, tau=0.05):
    """
    Combine the prior-implied returns with the views.

    The output is:
    - mu_bl: posterior expected returns
    - Sigma_bl: posterior covariance matrix
    """

    pi = np.asarray(pi)
    Sigma = np.asarray(Sigma)
    P = np.asarray(P)
    Q = np.asarray(Q)
    Omega = np.asarray(Omega)

    n_assets = Sigma.shape[0]
    n_views = P.shape[0]

    assert tau > 0, "tau must be positive."
    assert Sigma.shape == (n_assets, n_assets), "Sigma must be square."
    assert pi.shape == (n_assets,), "pi has the wrong shape."
    assert P.shape == (n_views, n_assets), "P has the wrong shape."
    assert Q.shape == (n_views,), "Q has the wrong shape."
    assert Omega.shape == (n_views, n_views), "Omega has the wrong shape."
    assert np.all(np.diag(Omega) > 0), "Omega diagonal values must be positive."

    # The small diagonal term helps avoid numerical inversion problems.
    Sigma = Sigma + 1e-8 * np.eye(n_assets)
    tau_sigma = tau * Sigma

    inv_tau_sigma = np.linalg.pinv(tau_sigma)
    inv_omega = np.linalg.pinv(Omega)

    posterior_cov_part = np.linalg.pinv(
        inv_tau_sigma + P.T @ inv_omega @ P
    )

    mu_bl = posterior_cov_part @ (
        inv_tau_sigma @ pi + P.T @ inv_omega @ Q
    )

    Sigma_bl = Sigma + posterior_cov_part
    Sigma_bl = (Sigma_bl + Sigma_bl.T) / 2

    assert not np.isnan(mu_bl).any(), "mu_bl contains NaN values."
    assert not np.isnan(Sigma_bl).any(), "Sigma_bl contains NaN values."

    return mu_bl, Sigma_bl

## **Rolling Black-Litterman backtest**



This function runs the Black–Litterman strategy in a rolling out-of-sample way. At each rebalance date, it uses only the previous 60 months of returns to estimate the covariance matrix, build the equal-weight neutral prior, generate the dynamic views, and compute the Black–Litterman posterior returns.

The posterior expected returns and covariance matrix are then used in a constrained max-Sharpe optimization. The portfolio is long-only, fully invested, has no leverage, and each ETF weight is capped at 20%.

After the weights are calculated, they are applied to the next month’s realized return. This no-look-ahead structure makes the Black–Litterman results comparable with the Markowitz and Michaud rolling backtests.


In [ ]:
def run_black_litterman_backtest(returns, tau=0.05, max_weight=0.20):
    """
    Run the dynamic Black-Litterman strategy using the same rolling logic
    as the previous Markowitz and Michaud backtests.
    """

    asset_list = pd.Index(returns.columns)
    n_assets = len(asset_list)

    assert max_weight * n_assets >= 1.0, "max_weight is too restrictive for a fully invested portfolio."

    bl_returns_list = []
    bl_weights_list = []
    bl_dates = []
    views_history = []

    latest_pi = None
    latest_mu_bl = None
    latest_Sigma_bl = None
    latest_views = None
    latest_w_eq = None

    for i in range(WINDOW, len(returns)):

        # This is the no-look-ahead structure:
        # estimate everything from the previous WINDOW months only.
        estimation_window = returns.iloc[i-WINDOW:i]
        next_month_return = returns.iloc[i]

        Sigma = estimation_window.cov().values * MONTHS_PER_YEAR
        Sigma = Sigma + 1e-8 * np.eye(n_assets)

        # Equal-weight neutral prior because historical ETF AUM is not available.
        w_eq = np.ones(n_assets) / n_assets

        # Reverse optimization gives prior-implied returns.
        pi = RISK_AVERSION * Sigma @ w_eq

        # Views are generated only from the past rolling window.
        P, Q, Omega, views_table = create_dynamic_individual_bl_views(
            estimation_window=estimation_window,
            asset_list=asset_list,
            Sigma=Sigma,
            pi=pi
        )

        mu_bl, Sigma_bl = black_litterman_posterior(
            pi=pi,
            Sigma=Sigma,
            P=P,
            Q=Q,
            Omega=Omega,
            tau=tau
        )

        weights = optimize_max_sharpe(
            expected_returns=mu_bl,
            cov_matrix=Sigma_bl,
            max_weight=max_weight,
            risk_free_rate=0.0
        )

        assert np.isclose(weights.sum(), 1.0, atol=1e-5), "Weights do not sum to 1."
        assert (weights >= -1e-8).all(), "Weights contain negative values."
        assert (weights <= max_weight + 1e-8).all(), "Weights exceed the maximum weight."

        portfolio_return = weights @ next_month_return.values

        current_date = returns.index[i]

        bl_returns_list.append(portfolio_return)
        bl_weights_list.append(weights)
        bl_dates.append(current_date)

        views_table = views_table.copy()
        views_table.insert(0, "Date", current_date)
        views_history.append(views_table)

        latest_pi = pi
        latest_mu_bl = mu_bl
        latest_Sigma_bl = Sigma_bl
        latest_views = views_table
        latest_w_eq = w_eq

    bl_returns = pd.Series(
        bl_returns_list,
        index=bl_dates,
        name="Black-Litterman"
    )

    bl_weights = pd.DataFrame(
        bl_weights_list,
        index=bl_dates,
        columns=returns.columns
    )

    views_history = pd.concat(views_history, ignore_index=True)

    bl_artifacts = {
        "views_history": views_history,
        "latest_views": latest_views,
        "latest_pi": latest_pi,
        "latest_mu_bl": latest_mu_bl,
        "latest_Sigma_bl": latest_Sigma_bl,
        "latest_w_eq": latest_w_eq
    }

    assert not bl_returns.isna().any(), "Black-Litterman returns contain NaN values."
    assert not bl_weights.isna().any().any(), "Black-Litterman weights contain NaN values."

    return bl_returns, bl_weights, bl_artifacts


bl_returns, bl_weights, bl_artifacts = run_black_litterman_backtest(
    returns=returns,
    tau=TAU,
    max_weight=MAX_WEIGHT
)

bl_returns.head()

In [ ]:
# Latest views generated by the model
latest_views_table = bl_artifacts["latest_views"]
latest_views_table.round(4)

In [ ]:
# How often each ETF was selected as the strongest or weakest momentum asset
views_history = bl_artifacts["views_history"]

relative_views_history = views_history[
    views_history["View type"] == "Relative"
].copy()

selection_frequency = pd.DataFrame({
    "Selected as Top Asset": relative_views_history["Top asset"].value_counts(),
    "Selected as Bottom Asset": relative_views_history["Bottom asset"].value_counts()
}).fillna(0).astype(int)

selection_frequency

In [ ]:

bl_performance_stats = performance_stats(bl_returns).round(2)
bl_performance_stats

In [ ]:
bl_growth = (1 + bl_returns).cumprod()

bl_growth.plot(
    figsize=(10, 5),
    title="Black-Litterman Cumulative Wealth"
)

plt.ylabel("Growth of 1 Dollar")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
average_bl_weights = bl_weights.mean().sort_values(ascending=False)

average_bl_weights_table = pd.DataFrame({
    "Average BL Weight (%)": average_bl_weights * 100
})

average_bl_weights_table.round(2)

In [ ]:
(average_bl_weights.sort_values() * 100).plot(
    kind="barh",
    figsize=(9, 7),
    title="Average Black-Litterman Portfolio Weights"
)

plt.xlabel("Average weight (%)")
plt.ylabel("ETF")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
bl_turnover = compute_turnover(bl_weights)

turnover_comparison_extended = pd.DataFrame({
    "Markowitz": markowitz_turnover,
    "Michaud": michaud_turnover,
    "Black-Litterman": bl_turnover
}).dropna()

turnover_comparison_extended.mean().round(4)

In [ ]:
turnover_comparison_extended.plot(
    figsize=(11, 5),
    title="Monthly Portfolio Turnover Including Black-Litterman"
)

plt.ylabel("Turnover")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

### Portfolio Turnover

This chart compares the monthly portfolio turnover of Markowitz, Michaud, and Black–Litterman. Turnover measures how much the portfolio weights change from one rebalance date to the next.

The Black–Litterman strategy has higher and more volatile turnover, because its views are updated dynamically every month based on recent ETF performance. When the selected views change, the posterior expected returns also change, so the optimizer may rebalance more aggressively.

Higher turnover can improve responsiveness, but it also implies higher trading costs. Therefore, this result should be interpreted together with performance, risk, and transaction-cost considerations.


In [ ]:
# Add Black-Litterman to the existing final comparison.
comparison_with_bl_returns = pd.DataFrame({
    "Equal Weight": equal_weight_returns,
    "60/40": benchmark_60_40_returns,
    "Markowitz Min Variance": markowitz_returns,
    "Michaud Resampled": michaud_returns,
    "Markowitz Max Sharpe": markowitz_sharpe_returns,
    "Black-Litterman": bl_returns
}).dropna()

comparison_with_bl_stats = comparison_with_bl_returns.apply(performance_stats).T.round(2)

comparison_with_bl_stats

In [ ]:
comparison_with_bl_growth = (1 + comparison_with_bl_returns).cumprod()

comparison_with_bl_growth.plot(
    figsize=(12, 6),
    title="Final Strategy Comparison Including Black-Litterman"
)

plt.ylabel("Growth of 1 Dollar")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
drawdowns_with_bl = comparison_with_bl_returns.apply(compute_drawdown)

drawdowns_with_bl.plot(
    figsize=(12, 6),
    title="Drawdown Comparison Including Black-Litterman"
)

plt.ylabel("Drawdown")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Tau sensitivity check
tau_values = [0.01, 0.05, 0.10]

tau_returns = {}

for tau_value in tau_values:
    tau_bl_returns, _, _ = run_black_litterman_backtest(
        returns=returns,
        tau=tau_value,
        max_weight=MAX_WEIGHT
    )

    tau_returns[f"tau={tau_value}"] = tau_bl_returns

tau_returns = pd.DataFrame(tau_returns).dropna()

tau_sensitivity_stats = tau_returns.apply(performance_stats).T.round(2)

tau_sensitivity_stats

In [ ]:
(1 + tau_returns).cumprod().plot(
    figsize=(11, 5),
    title="Black-Litterman Tau Sensitivity"
)

plt.ylabel("Growth of 1 Dollar")
plt.xlabel("Date")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Final validation checks
assert bl_weights.sum(axis=1).sub(1).abs().max() < 1e-5
assert (bl_weights >= -1e-8).all().all()
assert (bl_weights <= MAX_WEIGHT + 1e-8).all().all()
assert not bl_returns.isna().any()
assert not bl_weights.isna().any().any()

print("Black-Litterman checks passed.")
print("The backtest uses returns.iloc[i-WINDOW:i] to compute weights and applies them to returns.iloc[i].")

## **latest AUM-based portfolio**



This section is not part of the historical rolling backtest.  
It is an end-of-sample sensitivity analysis.

First, I build the latest AUM-based Black-Litterman allocation using the same automatic momentum-based views used before.  
Then, I allow the user to manually change the Black-Litterman views and observe how the final portfolio weights change live.

The latest AUM values are used only for the current/end-of-sample allocation, not for the historical backtest, because historical AUM was not available.

##**Latest AUM prior**



For the latest-date Black–Litterman diagnostic, I use the most recent ETF AUM values as the prior portfolio weights.

The AUM values were manually collected from FactSet for the selected ETFs. These values represent the latest available Assets Under Management for each ETF. Since only relative size matters, the AUM values can be entered in millions or billions, as long as the same unit is used for all ETFs.

The latest AUM weight of each ETF is calculated as:

AUM weight of ETF i = AUM of ETF i / Total AUM of all selected ETFs

So each ETF receives a prior weight proportional to its latest AUM.

This latest AUM prior is used only for the final-date diagnostic dashboard with manual views. It is not used in the historical 2008–2026 backtest, because using current AUM for past periods would create look-ahead bias.

In the historical rolling Black–Litterman backtest, I use an equal-weight prior instead. The latest AUM prior is only used to show how a more realistic current-size-based prior changes the latest Black–Litterman allocation.


In [ ]:
# Latest AUM values collected manually from FactSet.
# These are current values, so they are used only for the latest allocation.

import json
latest_aum_raw = json.loads((PROJECT_ROOT / "data/latest_aum.json").read_text())

latest_aum = pd.Series(latest_aum_raw).reindex(returns.columns)

latest_aum = latest_aum.apply(
    lambda x: float(str(x).replace(",", "").upper().replace("B", "")) * 1e9
    if str(x).upper().endswith("B")
    else float(str(x).replace(",", "").upper().replace("M", "")) * 1e6
)

latest_aum_weights = latest_aum / latest_aum.sum()

latest_aum_table = pd.DataFrame({
    "Latest AUM": latest_aum,
    "AUM Prior Weight (%)": latest_aum_weights * 100,
    "Asset Class": [asset_classes.get(ticker, "Not specified") for ticker in returns.columns]
}, index=returns.columns)

display(latest_aum_table.round(2))

print("Number of assets:", len(returns.columns))
print("Sum of AUM prior weights:", latest_aum_weights.sum().round(6))

In [ ]:
# This is the baseline latest allocation before the user changes views manually.

asset_list = pd.Index(returns.columns)

latest_estimation_window = returns.iloc[-WINDOW:]

Sigma_latest = latest_estimation_window.cov().values * MONTHS_PER_YEAR
Sigma_latest = Sigma_latest + 1e-8 * np.eye(len(asset_list))

prior_weights_latest = latest_aum_weights.values

# AUM-based prior-implied returns
pi_latest_aum = RISK_AVERSION * Sigma_latest @ prior_weights_latest

# Automatic views from the same momentum-based rule used before
P_auto, Q_auto, Omega_auto, latest_auto_views_table = create_dynamic_individual_bl_views(
    estimation_window=latest_estimation_window,
    asset_list=asset_list,
    Sigma=Sigma_latest,
    pi=pi_latest_aum
)

mu_auto_bl, Sigma_auto_bl = black_litterman_posterior(
    pi=pi_latest_aum,
    Sigma=Sigma_latest,
    P=P_auto,
    Q=Q_auto,
    Omega=Omega_auto,
    tau=TAU
)

weights_auto_bl = optimize_max_sharpe(
    expected_returns=mu_auto_bl,
    cov_matrix=Sigma_auto_bl,
    max_weight=MAX_WEIGHT,
    risk_free_rate=0.0
)

latest_auto_allocation = pd.DataFrame({
    "AUM Prior Weight (%)": prior_weights_latest * 100,
    "Auto BL Weight (%)": weights_auto_bl * 100,
    "Auto Change vs Prior (%)": (weights_auto_bl - prior_weights_latest) * 100,
    "Prior Return (%)": pi_latest_aum * 100,
    "Auto Posterior Return (%)": mu_auto_bl * 100,
    "Asset Class": [asset_classes.get(ticker, "Not specified") for ticker in asset_list]
}, index=asset_list)

print("Latest return window used for covariance:")
print(latest_estimation_window.index.min().date(), "to", latest_estimation_window.index.max().date())
print()

print("Automatic views generated from latest 12-month momentum:")
display(latest_auto_views_table.fillna("Not applicable").round(4))
print()

print("Latest AUM-based Black-Litterman allocation with automatic views:")
display(latest_auto_allocation.round(2))

print("Weight checks:")
print("Sum of weights:", weights_auto_bl.sum().round(6))
print("Min weight:", weights_auto_bl.min().round(6))
print("Max weight:", weights_auto_bl.max().round(6))

**latest AUM prior weights**
**vs**
**latest Black–Litterman posterior weights**

In [ ]:
latest_auto_allocation[
    ["AUM Prior Weight (%)", "Auto BL Weight (%)"]
].plot(
    kind="bar",
    figsize=(13, 6),
    title="Latest AUM Prior vs Automatic Black-Litterman Allocation"
)

plt.ylabel("Weight (%)")
plt.xlabel("ETF")
plt.grid(True)
plt.tight_layout()
plt.show()

##**Manual view matrix function**

In [ ]:
def create_manual_view_matrices(asset_list, Sigma, manual_views):
    asset_list = pd.Index(asset_list)
    n_assets = len(asset_list)

    P_rows = []
    Q_values = []
    omega_values = []
    view_records = []

    for view in manual_views:
        view_type = view["type"]

        if view_type == "Disabled":
            continue

        confidence = float(view["confidence"])
        q_value = float(view["q_pct"]) / 100

        if confidence <= 0 or confidence >= 1:
            raise ValueError("Confidence must be between 0 and 1.")

        if view_type == "Absolute":
            asset = view["asset_a"]

            P_row = np.zeros(n_assets)
            P_row[asset_list.get_loc(asset)] = 1.0

            view_name = f"{asset} expected annual return"

        elif view_type == "Relative":
            long_asset = view["asset_a"]
            short_asset = view["asset_b"]

            if long_asset == short_asset:
                raise ValueError("In a relative view, Asset A and Asset B must be different.")

            P_row = np.zeros(n_assets)
            P_row[asset_list.get_loc(long_asset)] = 1.0
            P_row[asset_list.get_loc(short_asset)] = -1.0

            view_name = f"{long_asset} outperforms {short_asset}"

        view_variance = float(P_row @ Sigma @ P_row.T)
        view_variance = max(view_variance, 1e-8)

        omega_value = ((1 - confidence) / confidence) * view_variance
        omega_value = max(float(omega_value), 1e-6)

        P_rows.append(P_row)
        Q_values.append(q_value)
        omega_values.append(omega_value)

        view_records.append({
            "View Type": view_type,
            "View": view_name,
            "Annual View Q (%)": q_value * 100,
            "Confidence": confidence,
            "Omega": omega_value
        })

    if len(P_rows) == 0:
        return None, None, None, pd.DataFrame()

    P = np.vstack(P_rows)
    Q = np.array(Q_values)
    Omega = np.diag(omega_values)

    views_table = pd.DataFrame(view_records)

    return P, Q, Omega, views_table

In [ ]:
# The interactive panel starts from the automatic views.
# The user can then change the view type, assets, view percentage, and confidence.

auto_relative_view = latest_auto_views_table[
    latest_auto_views_table["View type"] == "Relative"
].iloc[0]

auto_absolute_view = latest_auto_views_table[
    latest_auto_views_table["View type"] == "Absolute"
].iloc[0]

default_top_asset = auto_relative_view["Top asset"]
default_bottom_asset = auto_relative_view["Bottom asset"]

default_relative_view_pct = auto_relative_view["Q annual view"] * 100
default_relative_confidence = auto_relative_view["Confidence"]

default_absolute_asset = auto_absolute_view["Top asset"]
default_absolute_view_pct = auto_absolute_view["Q annual view"] * 100
default_absolute_confidence = auto_absolute_view["Confidence"]

print("Default manual-view starting point based on automatic views:")
print("Relative view:", default_top_asset, "outperforms", default_bottom_asset)
print("Relative view Q (%):", round(default_relative_view_pct, 2))
print("Relative confidence:", default_relative_confidence)
print()
print("Absolute view asset:", default_absolute_asset)
print("Absolute view Q (%):", round(default_absolute_view_pct, 2))
print("Absolute confidence:", default_absolute_confidence)

In [ ]:
# Import tools for the interactive Black-Litterman manual view panel

import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
asset_options = list(asset_list)

view_widgets = []

default_view_types = ["Relative", "Absolute", "Disabled"]
default_asset_a_values = [default_top_asset, default_absolute_asset, default_top_asset]
default_asset_b_values = [default_bottom_asset, default_bottom_asset, default_bottom_asset]
default_q_values = [default_relative_view_pct, default_absolute_view_pct, 0.0]
default_conf_values = [default_relative_confidence, default_absolute_confidence, 0.60]

for i in range(3):
    view_type_widget = widgets.Dropdown(
        options=["Disabled", "Absolute", "Relative"],
        value=default_view_types[i],
        description=f"View {i+1}:",
        layout=widgets.Layout(width="260px")
    )

    asset_a_widget = widgets.Dropdown(
        options=asset_options,
        value=default_asset_a_values[i],
        description="Asset A:",
        layout=widgets.Layout(width="260px")
    )

    asset_b_widget = widgets.Dropdown(
        options=asset_options,
        value=default_asset_b_values[i],
        description="Asset B:",
        layout=widgets.Layout(width="260px")
    )

    q_widget = widgets.FloatText(
        value=round(float(default_q_values[i]), 2),
        description="View %:",
        layout=widgets.Layout(width="260px")
    )

    confidence_widget = widgets.FloatSlider(
        value=float(default_conf_values[i]),
        min=0.50,
        max=0.95,
        step=0.05,
        description="Confidence:",
        readout_format=".2f",
        layout=widgets.Layout(width="330px")
    )

    view_widgets.append({
        "type": view_type_widget,
        "asset_a": asset_a_widget,
        "asset_b": asset_b_widget,
        "q_pct": q_widget,
        "confidence": confidence_widget
    })


controls_children = []

for i, vw in enumerate(view_widgets):
    controls_children.append(
        widgets.HTML(
            value=f"""
            <b>View slot {i+1}</b><br>
            Absolute: Asset A expected annual return<br>
            Relative: Asset A outperforms Asset B
            """
        )
    )

    controls_children.extend([
        vw["type"],
        vw["asset_a"],
        vw["asset_b"],
        vw["q_pct"],
        vw["confidence"],
        widgets.HTML(value="<hr>")
    ])

controls_box = widgets.VBox(
    controls_children,
    layout=widgets.Layout(
        width="42%",
        border="1px solid lightgray",
        padding="10px"
    )
)

output_box = widgets.Output(
    layout=widgets.Layout(
        width="58%",
        border="1px solid lightgray",
        padding="10px"
    )
)

**Live update function and display**

In [ ]:
def update_manual_bl_allocation(change=None):
    with output_box:
        clear_output(wait=True)

        manual_views = []

        for vw in view_widgets:
            manual_views.append({
                "type": vw["type"].value,
                "asset_a": vw["asset_a"].value,
                "asset_b": vw["asset_b"].value,
                "q_pct": vw["q_pct"].value,
                "confidence": vw["confidence"].value
            })

        # This section always uses the latest AUM-based prior.
        prior_weights = prior_weights_latest
        pi_manual = pi_latest_aum

        try:
            P_manual, Q_manual, Omega_manual, manual_views_table = create_manual_view_matrices(
                asset_list=asset_list,
                Sigma=Sigma_latest,
                manual_views=manual_views
            )

            if P_manual is None:
                mu_manual_bl = pi_manual
                Sigma_manual_bl = Sigma_latest
                active_view_text = "No active manual view selected. Using only the latest AUM-based prior."

            else:
                mu_manual_bl, Sigma_manual_bl = black_litterman_posterior(
                    pi=pi_manual,
                    Sigma=Sigma_latest,
                    P=P_manual,
                    Q=Q_manual,
                    Omega=Omega_manual,
                    tau=TAU
                )

                active_view_text = "Active manual views: " + " | ".join(
                    manual_views_table["View"].astype(str).tolist()
                )

            weights_manual_bl = optimize_max_sharpe(
                expected_returns=mu_manual_bl,
                cov_matrix=Sigma_manual_bl,
                max_weight=MAX_WEIGHT,
                risk_free_rate=0.0
            )

            manual_weights_series = pd.Series(
                weights_manual_bl * 100,
                index=asset_list,
                name="Weight (%)"
            ).round(2)

            print(active_view_text)
            print("Latest allocation only: AUM-based prior + manual views")
            print("Sum of weights:", weights_manual_bl.sum().round(6))
            print("Max weight:", weights_manual_bl.max().round(6))
            print()

            # Make a compact table: 4 blocks of ETF + weight, so it is not too crowded.
            weight_table = manual_weights_series.reset_index()
            weight_table.columns = ["ETF", "Weight (%)"]

            blocks = []
            block_size = 5

            for start in range(0, len(weight_table), block_size):
                block = weight_table.iloc[start:start + block_size].reset_index(drop=True)
                block.columns = [f"ETF {len(blocks) + 1}", f"Weight {len(blocks) + 1} (%)"]
                blocks.append(block)

            compact_weight_table = pd.concat(blocks, axis=1)

            fig, (ax_chart, ax_table) = plt.subplots(
                nrows=2,
                figsize=(13, 8),
                gridspec_kw={"height_ratios": [3.5, 1.2]}
            )

            manual_weights_series.plot(
                kind="bar",
                ax=ax_chart,
                title="Latest AUM-Based Black-Litterman Allocation with Manual Views"
            )

            ax_chart.set_ylabel("Portfolio Weight (%)")
            ax_chart.set_xlabel("ETF")
            ax_chart.grid(True)

            ax_table.axis("off")

            table = ax_table.table(
                cellText=compact_weight_table.values,
                colLabels=compact_weight_table.columns,
                cellLoc="center",
                loc="center"
            )

            table.auto_set_font_size(False)
            table.set_fontsize(9)
            table.scale(1, 1.25)

            plt.tight_layout()
            plt.show()

        except Exception as error:
            print("There is a problem with the selected views:")
            print(error)


for vw in view_widgets:
    for widget in vw.values():
        widget.observe(update_manual_bl_allocation, names="value")

display(widgets.HBox([controls_box, output_box]))

update_manual_bl_allocation()